# Imagery overlays

`add_imagery` drops a georeferenced raster — a GeoTIFF, a COG, anything GDAL
reads — onto the map. The raster is **reprojected into the map's own CRS** on
the way in, so a source in any projection lands exactly where it belongs;
nodata pixels come out transparent; a single-band raster colours through the
same colormaps `color_col` uses.

Needs `rasterio` (`pip install rasterio`), swiftmap's one optional dependency.
This notebook authors its own rasters in-cell, so there is nothing to download.

In [ ]:
import numpy as np
import rasterio
from rasterio.transform import from_bounds

# A synthetic single-band "elevation" field over the Strait of Gibraltar:
# two gaussian ridges on a gradient, with a nodata notch in one corner.
W, H = 400, 300
west, south, east, north = -5.75, 35.85, -5.15, 36.25
ys, xs = np.mgrid[0:H, 0:W]
field = (
    900 * np.exp(-(((xs - 120) / 60) ** 2 + ((ys - 90) / 40) ** 2))
    + 1400 * np.exp(-(((xs - 300) / 50) ** 2 + ((ys - 210) / 55) ** 2))
    + 2.0 * ys
).astype("float32")
field[:60, :80] = -9999.0                      # nodata notch

with rasterio.open(
    "demo_dem.tif", "w", driver="GTiff", width=W, height=H, count=1,
    dtype="float32", crs="EPSG:4326", nodata=-9999.0,
    transform=from_bounds(west, south, east, north, W, H),
) as dst:
    dst.write(field, 1)
"demo_dem.tif written"

## One call

The file's CRS is EPSG:4326; the map's is web-mercator. Nobody has to care —
the warp happens on the way in, and auto-fit frames the result:

In [ ]:
from swiftmap import Map

m = Map()
m.add_imagery("demo_dem.tif", name="Synthetic DEM",
              colormap="turbo", vmin=0, vmax=1500, opacity=0.85)
m

The nodata notch is transparent — the basemap shows through. `vmin`/`vmax` fix
the ramp's endpoints exactly as they do for `color_col`; RGB(A) rasters skip
the colormap and keep their own pixels.

## It sits under your vectors

Imagery is context: it draws above the basemap, below every vector layer, and
it never answers clicks — features and the coordinate readout stay in charge.

In [ ]:
import pandas as pd

rng = np.random.default_rng(8)
n = 60
sites = pd.DataFrame({
    "lat": 36.05 + rng.normal(0, 0.05, n),
    "lon": -5.45 + rng.normal(0, 0.09, n),
    "reading": np.round(rng.gamma(4, 4, n), 1),
})
m.add_circle_markers(sites, name="Sites", color_col="reading");

## Projection is absorbed, not trusted to luck

The same field authored in a *projected* CRS — UTM zone 30N — lands on the
identical footprint, because every source is warped into the map's CRS rather
than pinned by corner coordinates and hope. Overlaying both at half opacity
shows one shape, not two:

In [ ]:
from rasterio.warp import transform_bounds

utm_bounds = transform_bounds("EPSG:4326", "EPSG:32630",
                              west, south, east, north)
with rasterio.open(
    "demo_dem_utm.tif", "w", driver="GTiff", width=W, height=H, count=1,
    dtype="float32", crs="EPSG:32630", nodata=-9999.0,
    transform=from_bounds(*utm_bounds, W, H),
) as dst:
    dst.write(field, 1)

m.add_imagery("demo_dem_utm.tif", name="Same field, UTM 30N",
              colormap="greys", vmin=0, vmax=1500, opacity=0.5);

Toggle the two imagery layers in the sidebar against each other: same
footprint, same ridges. (The UTM copy is rectangular in *its* CRS, so its
warped edges pick up the slight tilt of honesty — corners land where the
projection says, not where a bounding box guesses.)

## Everything else comes for free

Sidebar toggles, folders, `opacity`, removal, and the static export all treat
imagery as an ordinary layer — an exported map (**08_export**) carries its
rasters inside the file. Large sources are downsampled to `max_size=2048` on
the longest edge on the way in; pass a bigger `max_size` when detail beats
payload.